##### Copyright 2025 Google LLC.

In [12]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Email Concierge Bot: A Function Calling Email Organizer

### What This Notebook Does:
- **Classifies Emails Using Text Embeddings:**  
  Leverages text embedding techniques to analyze email content and determine its category (e.g., deadlines, spam, meeting invites).

- **Triggers Automated Actions via Function Calling:**  
  Dynamically calls pre-defined functions (such as marking emails as important, deleting spam, or scheduling calendar events) based on the email classification.

- **Demonstrates Adaptability:**  
  Showcases how Gemini can seamlessly integrate text embedding and function calling to handle various email scenarios in a flexible, automated manner.

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Email_Concierge_Function_Calling_Bot.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/>

In [13]:
%pip install -qU "google-genai>=1.0.0"

To run this notebook, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you are running in a different environment, you can store your key in an environment variable. See [Authentication](../quickstarts/Authentication.ipynb) to learn more.

In [14]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

## Define the API

This smart email organizer leverages advanced text embedding techniques and dynamic function calling to intelligently manage your email workflow. By analyzing the semantic content of your emails via embeddings, the bot classifies messages into categories such as "important" or "not_important". Based on these classifications, it then automatically triggers the appropriate action—whether that's prioritizing an email, flagging it for follow-up, or updating its category.

The API maintains global state through variables such as:
- **`inbox_emails`**: A list that stores incoming emails as dictionaries (with keys like `"content"` and `"category"`).
- **`category_centroids`**: A dictionary that holds normalized centroids for each predefined email category, computed from example emails in `reference_emails`.

Each function call not only updates the email status but also, once an email is processed, moves it from `inbox_emails` and triggers the necessary action. Moreover, the system can learn from corrections: when you update an email's category via the `learn_email` function, it incorporates the new example into its training data and recalculates the centroids, thereby continuously improving its classification accuracy.

Below is the complete API definition:


In [15]:
import numpy as np
from google.genai import types
import logging
from typing import Dict, Optional, List


SIMILARITY_THRESHOLD = 0.7
EMBEDDING_MODEL_ID = 'models/embedding-001'
inbox_emails = []

# Reference emails for centroid computation
reference_emails = {
    "important": [
        "Action required: confirm your subscription",
        "Meeting canceled: Project Alpha sync",
        "Invoice due: payment reminder for INV-123",
        "Urgent security alert for your account"
    ],
    "not_important": [
        "Weekly tech newsletter digest",
        "Summer sale: Up to 50% off!",
        "John Doe mentioned you on SocialApp",
        "Your monthly reading suggestions"
    ]
}


category_centroids = {}

def get_embedding(text: str) -> Optional[np.array]:
    """
    Fetches the embedding for the given text using the latest API call.
    Returns a NumPy array of embedding values, or None if an error occurs.

    Uses the client to call the embed_content method with task_type "retrieval_document".
    """
    if not text:
        logging.warning("Empty text provided for embedding.")
        return None

    try:
        response = client.models.embed_content(
            model=EMBEDDING_MODEL_ID,
            contents=text,
            config=types.EmbedContentConfig(
                task_type="retrieval_document"
            )
        )
        embedding_values = response.embeddings[0].values
        return np.array(embedding_values)
    except Exception as e:
        logging.error(f"Error getting embedding for text '{text[:50]}...': {e}")
        return None

def initialize_category_centroids() -> Dict[str, np.array]:
    """
    Computes the normalized centroid (mean embedding) for each predefined email category.

    For each category in reference_emails, it:
      - Computes embeddings for each example email.
      - Averages these embeddings to get a centroid.
      - Normalizes the centroid vector.

    Returns:
        A dictionary mapping category names to their normalized centroid vectors.
    """
    global category_centroids
    centroids = {}
    for category, examples in reference_emails.items():
        embeddings = [get_embedding(ex) for ex in examples]
        valid_embeddings = [emb for emb in embeddings if emb is not None]

        if not valid_embeddings:
            logging.warning(f"No valid embeddings for category '{category}'.")
            continue

        try:
            centroid = np.mean(np.vstack(valid_embeddings), axis=0)
        except Exception as e:
            logging.error(f"Error computing centroid for '{category}': {e}")
            continue

        norm = np.linalg.norm(centroid)
        if norm < 1e-6:
            logging.warning(f"Centroid norm too low for '{category}'.")
            continue

        centroids[category] = centroid / norm
        logging.info(f"Created centroid for '{category}' using {len(valid_embeddings)} examples.")

    logging.info(f"Initialized centroids for {len(centroids)} categories.")
    category_centroids = centroids
    return centroids

# Initialize centroids at startup
initialize_category_centroids()

def classify_email_using_centroids(email_body: str) -> Optional[str]:
    """
    Classifies an email by computing its embedding and comparing it against precomputed normalized centroids.

    It calculates cosine similarity (via dot product) between the normalized email embedding
    and each category centroid. If the highest similarity is greater than or equal to
    SIMILARITY_THRESHOLD, the corresponding category is returned.

    Args:
        email_body: The email content to classify.

    Returns:
        The category name (e.g., "important" or "not_important") if the similarity is high enough;
        otherwise, returns None.
    """
    global category_centroids
    if not category_centroids:
        logging.warning("No centroids available for classification.")
        return None

    email_embedding = get_embedding(email_body)
    if email_embedding is None:
        logging.info(f"Could not compute embedding for email: '{email_body[:50]}...'.")
        return None

    email_norm = np.linalg.norm(email_embedding)
    if email_norm < 1e-6:
        logging.warning(f"Email embedding norm too low for: '{email_body[:50]}...'.")
        return None

    normalized_email = email_embedding / email_norm

    similarities = {
        category: np.dot(normalized_email, centroid)
        for category, centroid in category_centroids.items()
    }
    best_category = max(similarities, key=similarities.get)
    best_similarity = similarities[best_category]

    logging.debug(f"Email: '{email_body[:30]}...' best match: '{best_category}' with similarity {best_similarity:.4f}")

    return best_category if best_similarity >= SIMILARITY_THRESHOLD else None

def add_as_important(email_content: str):
    """
    Appends the given email content to the inbox as 'important'.

    The function records the email with a category label of "important".
    """
    inbox_emails.append({"content": email_content, "category": "important"})
    logging.info("Email added as important.")

def add_as_not_important(email_content: str):
    """
    Appends the given email content to the inbox as 'not_important'.

    The function records the email with a category label of "not_important".
    """
    inbox_emails.append({"content": email_content, "category": "not_important"})
    logging.info("Email added as not important.")

def get_all_emails() -> List[Dict]:
    """
    Returns the complete list of emails currently stored in the inbox.
    Each email is expected to be a dictionary containing keys like 'content' and 'category'.
    """
    return inbox_emails

def learn_email(email_content: str, correct_category: str):
    """
    Learns from a new email example by updating our reference emails and retraining the centroids.

    Args:
        email_content: The new email text to learn from.
        correct_category: The correct category for this email (e.g., "important" or "not_important").

    Process:
        1. Adds the email to the reference_emails list for the given category.
        2. Recomputes the category centroids by calling initialize_category_centroids.
        3. Logs the learning action for future reference.
    """

    if correct_category not in reference_emails:
        reference_emails[correct_category] = []

    reference_emails[correct_category].append(email_content)
    logging.info(f"New training example added for category '{correct_category}'.")

    initialize_category_centroids()
    logging.info(f"Centroids updated after learning new email.")


## Test the API

With the functions written, test that they work as expected.

In [16]:
important_email = "Urgent: Please review the attached report immediately"
not_important_email = "Notice: Free Subscription to newletter"


add_as_important(important_email)
add_as_not_important(not_important_email)

print("\n--- Global State ---")
print("Processed Emails:", inbox_emails)


--- Global State ---
Processed Emails: [{'content': 'Urgent: Please review the attached report immediately', 'category': 'important'}, {'content': 'Notice: Free Subscription to newletter', 'category': 'not_important'}]


## Set up the model

In this step you collate the functions into a "system" that is passed as `tools`, instantiate the model and start the chat session.

A retriable `send_message` function is also defined to help with low-quota conversations.

In [17]:
email_tools = [
    classify_email_using_centroids,
    add_as_important,
    add_as_not_important,
    get_all_emails,
    learn_email
]

inbox_emails=[]

MODEL_ID = "gemini-2.0-flash-lite"  # @param ["gemini-2.0-flash-lite","gemini-2.0-flash","gemini-2.5-pro-exp-03-25"] {"allow-input":true, isTemplate: true}

chat = client.chats.create(
    model=MODEL_ID,
    config=types.GenerateContentConfig(
        tools=email_tools,
        system_instruction = """
          1. When processing an email, first call classify_email_using_centroids(email_body) to determine its category. If it returns 'important' or 'not_important', immediately call add_as_important(email_content) or add_as_not_important(email_content) respectively. If no valid classification is returned, analyze the email and choose the appropriate function. Respond with only the function call (including the email content) that you are executing.
          2. Whenever a user asks for a specific email (e.g., 'find me the email about the assignment'), you must first call get_all_emails to retrieve the full list of emails. Then, analyze the content of these emails to locate the one that best matches the user's request before returning the result.
          3. If a user instructs you to change the category of an email (e.g., 'mark the email about the meeting as important'), you should pass the entire email content along with the new category to the learn_email function. This ensures that the system learns from the correction and updates its centroid-based classification accordingly or If the user's instruction to change an email's category does not reference a specific email, you should call get_all_emails to retrieve all emails, search through them for those that match the description provided in the request, and then for each matching email, pass its content along with the requested category to the learn_email function. In all cases, respond with a confirmation message without asking for additional input.
        """
    )
)

def process_email_with_gemini(email_content: str):
    response = chat.send_message(f"""
      I am providing an email to classify using our centroid-based system.
      Email:
      {email_content}
    """)

    print(response.text)


### Loading Dataset

In [18]:
import requests

url = "https://gist.githubusercontent.com/andycandy/7fc329566d9e1d6a63b6db56b2700219/raw/"

try:
    response = requests.get(url)
    response.raise_for_status()
    dummy_emails = response.json()
except Exception  as e:
    print(f"Error : {e}")

## Simulate recieving a few emails

In [19]:
import random

sample_emails = random.sample(dummy_emails, 5)

for email in sample_emails:
    email_string = (
        f"Subject: {email.get('subject', '')}\n"
        f"Body: {email.get('body', '')}"
    )
    print(email_string)
    process_email_with_gemini(email_string)


Subject: Act Now! Earn $5000 Weekly from Home
Body: Join our work-from-home program and earn $5000 per week. No experience required!
OK. I've marked that email as not important.

Subject: Academic Advising Appointment Reminder
Body: This is a reminder for your academic advising appointment scheduled for next Monday.
OK. I've marked that email as important.

Subject: Congratulations! You've won a free iPhone
Body: Claim your free iPhone now! Click here to redeem your prize.
OK. I've marked that email as not important.

Subject: Limited Offer: Free Smartphone Upgrade
Body: Upgrade your smartphone for free! Limited offer, first come first served. Click to learn more.
OK. I've marked that email as not important.

Subject: Graduation Ceremony Details Announcement
Body: Important details regarding the graduation ceremony have been released. Please review the schedule and guidelines.
OK. I've marked that email as important.



# Chat with Email Classification Learning Bot
With our email classification system defined and the Gemini chat session created, all that's left is to connect user input to the model and display the output in a loop. This interactive loop continues until you decide to exit by providing an empty input or typing "exit"/"quit".

When run in Colab or a similar environment, any fixed-width text (like prompts or print statements) originates from your Python code, while regular text comes from the Gemini API. User inputs are displayed in outlined boxes with a leading ">".

Try it out and modify the classification to directly teach the bot what to do and what not to do—update any misclassified emails by correcting their categories with your command!

In [20]:
from IPython.display import display, Markdown

print("Welcome to Email Classification Learning Bot!")
print("Fix wrong classification of emails by updating their category with your command.")
print("Chat with bot or leave input empty to quit:\n")

while True:
    user_input = input("> ")
    if not user_input.strip() or user_input.lower() in ["exit", "quit"]:
        break
    response = chat.send_message(user_input)
    display(Markdown(response.text))

print("\n[Email Classification Learning Bot session over]")


Welcome to Email Classification Learning Bot!
Fix wrong classification of emails by updating their category with your command.
Chat with bot or leave input empty to quit:

> What emails have I got?


Here are the emails you have:

*   Subject: Act Now! Earn $5000 Weekly from Home
    Body: Join our work-from-home program and earn $5000 per week. No experience required!
    Category: not\_important
*   Subject: Academic Advising Appointment Reminder
    Body: This is a reminder for your academic advising appointment scheduled for next Monday.
    Category: important
*   Subject: Congratulations! You've won a free iPhone
    Body: Claim your free iPhone now! Click here to redeem your prize.
    Category: not\_important
*   Subject: Limited Offer: Free Smartphone Upgrade
    Body: Upgrade your smartphone for free! Limited offer, first come first served. Click to learn more.
    Category: not\_important
*   Subject: Graduation Ceremony Details Announcement
    Body: Important details regarding the graduation ceremony have been released. Please review the schedule and guidelines.
    Category: important


> Don't mark any easy income related mails not important


OK. I've updated the system.


> 

[Email Classification Learning Bot session over]


Try it out and fix any misclassified emails by updating their categories with your command!

Some things to try:

* Ask for a specific email (e.g., "find me the email about the assignment")
* Use terms that are not explicitly mentioned in the prompt (e.g., "student portal mails are urgent" or "newsletter emails are not important")
* Change your mind mid-command (e.g., "uhh, cancel that, mark the email as important instead")
* Go off-script (e.g., "what are the upcoming project updates?")

In [21]:
process_email_with_gemini("Subject: Don't Miss Out! Generate Passive Income of $3000 Weekly Online Body: Discover our revolutionary online system that allows you to generate $3000 in passive income every week. No prior skills needed!")

OK. I've marked that email as important.



## See also

This sample app showed you how to integrate a traditional software system (the coffee ordering functions) and an AI agent powered by the Gemini API. This is a simple, practical way to use LLMs that allows for open-ended human language input and output that feels natural, but still keeps a human in the loop to ensure correct operation.

To learn more about how Barista Bot works, check out:

* The [Barista Bot](https://aistudio.google.com/app/prompts/barista-bot) prompt
* [System instructions](../quickstarts/System_instructions.ipynb)
* [Automatic function calling](../quickstarts/Function_calling.ipynb)
